# Preprocessing & Feature Engineering (Stunting)
Notebook ini bertugas untuk:
1. Membaca file Excel Posyandu mentah (`data/raw/`).
2. Membuang kolom yang tidak relevan (No, Nama, Alamat, OrangTua) demi privasi dan mencegah model bias.
3. Menambahkan **Feature Engineering** (Menghitung Pertumbuhan Kenaikan BB/TB & Rasio BB/TB berdasar standar WHO).
4. Membangkitkan (generate) ratusan data *dummy* secara logis berdasarkan rata-rata sebaran data asli.
5. Menyimpan data gabungan (asli + dummy) ke `data/processed/` siap untuk dilatih

In [8]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

In [9]:
# Load Data Mentah & Perbaiki Format Header
raw_data_path = '../data/raw/dataset_posyandu.xlsx'

# Mengabaikan Baris 1 dan 2 Sebagai Header
df_asli = pd.read_excel(raw_data_path, header=[0, 1])

# Meratakan Judul Bertingkat Menjadi Satu Baris (Contoh: 'January_BB(kg)')
new_columns = []
for col in df_asli.columns:
    col_0 = str(col[0]).strip()
    col_1 = str(col[1]).strip()
    if 'Unnamed' in col_0: col_0 = ''
    if 'Unnamed' in col_1: col_1 = ''
    
    if col_0 and col_1:
        new_name = f"{col_0}_{col_1}"
    else:
        new_name = col_0 or col_1
    new_columns.append(new_name)

df_asli.columns = new_columns
print("Kolom berhasil dirapikan!")
df_asli.head(2)

Kolom berhasil dirapikan!


,No,Nama,Umur (Bulan),Jenis Kelamin,OrangTua,Alamat (Dusun),January_BB(kg),January_TB(cm),January_LL(cm),January_LK(cm),...,March_LK(cm),April_BB(kg),April_TB(cm),April_LL(cm),April_LK(cm),May_BB(kg),May_TB(cm),May_LL(cm),May_LK(cm),NORMAL(0) STUNTING (1)
0,1,CHAIRUNISA SALSABILA PUTRI,27,P,NaN,PAHING,10.8,81.7,16.0,45.0,...,46.0,10.8,84.0,14.0,48.0,11.2,86.0,15.0,49.0,0
1,2,DEANDRA ALFATHUNISA SYAFAZEA,36,P,NaN,PAHING,10.3,83.6,14.0,47.5,...,47.0,10.8,86.8,14.0,46.0,11.0,87.0,15.0,47.0,0


In [10]:
# 2. Membersihkan Kolom Pengenal, Encoding Gender, & Paksa Tipe Data Numerik
kolom_dihapus = ['No', 'Nama', 'OrangTua', 'Alamat (Dusun)']
kolom_drop_aman = [col for col in kolom_dihapus if col in df_asli.columns]
df_bersih = df_asli.drop(columns=kolom_drop_aman).copy()

if 'Jenis Kelamin' in df_bersih.columns:
    df_bersih['Jenis Kelamin'] = df_bersih['Jenis Kelamin'].map({'L': 0, 'P': 1})

# [SOLUSI ERROR] Terkadang kader Posyandu mengetik strip '-' atau spasi pada sel Excel yang kosong.
# Baris di bawah ini akan MEMAKSA secara agresif seluruh kolom tersisa menjadi angka murni (Float/Int).
# Jika ada teks nyasar, otomatis akan diubah menjadi kosong (NaN) agar aman saat dirata-rata.
df_bersih = df_bersih.apply(pd.to_numeric, errors='coerce')

df_bersih.head(2)

,Umur (Bulan),Jenis Kelamin,January_BB(kg),January_TB(cm),January_LL(cm),January_LK(cm),February_BB(kg),February_TB(cm),February_LL(cm),February_LK(cm),...,March_LK(cm),April_BB(kg),April_TB(cm),April_LL(cm),April_LK(cm),May_BB(kg),May_TB(cm),May_LL(cm),May_LK(cm),NORMAL(0) STUNTING (1)
0,27,1,10.8,81.7,16.0,45.0,11.1,82.4,15.5,45.0,...,46.0,10.8,84.0,14.0,48.0,11.2,86.0,15.0,49.0,0
1,36,1,10.3,83.6,14.0,47.5,10.4,83.6,14.0,48.0,...,47.0,10.8,86.8,14.0,46.0,11.0,87.0,15.0,47.0,0


In [11]:
# ==========================================
# FEATURE ENGINEERING (Kolom Baru)
# ==========================================
try:
    # Menghitung Berapa Banyak BB dan TB Bertambah Dari Januari ke Mei
    df_bersih['Delta_BB(kg)'] = df_bersih['May_BB(kg)'] - df_bersih['January_BB(kg)']
    df_bersih['Delta_TB(cm)'] = df_bersih['May_TB(cm)'] - df_bersih['January_TB(cm)']
    
    # Menghitung Rasio Berat Terhadap Tinggi di Bulan Terakhir
    df_bersih['Rasio_BB_TB_Akhir'] = df_bersih['May_BB(kg)'] / (df_bersih['May_TB(cm)'] + 0.001)
    
    print("Feature Engineering Berhasil!")
except KeyError as e:
    print(f"Error: Kolom tidak ditemukan {e}")

df_bersih.head(2)

Feature Engineering Berhasil!


,Umur (Bulan),Jenis Kelamin,January_BB(kg),January_TB(cm),January_LL(cm),January_LK(cm),February_BB(kg),February_TB(cm),February_LL(cm),February_LK(cm),...,April_LL(cm),April_LK(cm),May_BB(kg),May_TB(cm),May_LL(cm),May_LK(cm),NORMAL(0) STUNTING (1),Delta_BB(kg),Delta_TB(cm),Rasio_BB_TB_Akhir
0,27,1,10.8,81.7,16.0,45.0,11.1,82.4,15.5,45.0,...,14.0,48.0,11.2,86.0,15.0,49.0,0,0.4,4.3,0.130231
1,36,1,10.3,83.6,14.0,47.5,10.4,83.6,14.0,48.0,...,14.0,46.0,11.0,87.0,15.0,47.0,0,0.7,3.4,0.126435


In [12]:
# Fungsi Pembuat Data Dummy
def generate_synthetic_data(df, target_col, num_samples=1000):
    synthetic_rows = []
    class_counts = df[target_col].value_counts(normalize=True)
    
    for class_val, prop in class_counts.items():
        n_class_samples = int(num_samples * prop)
        df_class = df[df[target_col] == class_val]
        
        synthetic_class_data = {}
        for col in df.columns:
            if col == target_col:
                synthetic_class_data[col] = [class_val] * n_class_samples
            elif col == 'Jenis Kelamin':
                synthetic_class_data[col] = np.random.choice([0, 1], size=n_class_samples)
            elif col == 'Umur(bulan)' or col == 'Umur (Bulan)':
                min_umur, max_umur = df_class[col].min(), df_class[col].max()
                if pd.isna(min_umur): min_umur=1
                if pd.isna(max_umur): max_umur=60
                synthetic_class_data[col] = np.random.randint(min_umur, max_umur + 1, size=n_class_samples)
            else:
                mean_val = df_class[col].mean()
                std_val = df_class[col].std()
                if pd.isna(std_val) or std_val == 0: std_val = 0.5
                if pd.isna(mean_val): mean_val = 0
                gen_data = np.random.normal(loc=mean_val, scale=std_val, size=n_class_samples)
                gen_data = np.clip(gen_data, a_min=0.1, a_max=None)
                synthetic_class_data[col] = np.round(gen_data, 2)
                
        synthetic_rows.append(pd.DataFrame(synthetic_class_data))
    return pd.concat(synthetic_rows, ignore_index=True)

target_col = [col for col in df_bersih.columns if 'STUNTING' in col.upper() or 'NORMAL' in col.upper()]
if target_col:
    target_col = target_col[0]
    df_dummy = generate_synthetic_data(df_bersih, target_col, num_samples=920)
    print(f"Berhasil membuat {len(df_dummy)} data dummy!")

Berhasil membuat 919 data dummy!


In [13]:
# Gabungkan & Simpan Data Final
if 'df_dummy' in locals():
    df_gabungan = pd.concat([df_bersih, df_dummy], ignore_index=True)
    df_gabungan = df_gabungan.sample(frac=1, random_state=42).reset_index(drop=True)
    
    processed_data_path = '../data/processed/dataset_final_training.csv' 
    os.makedirs('../data/processed', exist_ok=True)
    df_gabungan.to_csv(processed_data_path, index=False)
    print(f"[SUKSES] Data final tersimpan di: {processed_data_path}\nTotal Baris: {len(df_gabungan)}")

[SUKSES] Data final tersimpan di: ../data/processed/dataset_final_training.csv
Total Baris: 1009
